In [36]:
import warnings
warnings.filterwarnings('ignore')

In [37]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import ast
from sklearn.cluster import KMeans

from rapidfuzz import fuzz, process
import unicodedata

import joblib
import re

In [38]:
pd.set_option('display.max_columns', 100)

In [39]:
us_state_to_abbrev = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
    "District of Columbia": "DC",
    "American Samoa": "AS",
    "Guam": "GU",
    "Northern Mariana Islands": "MP",
    "Puerto Rico": "PR",
    "United States Minor Outlying Islands": "UM",
    "Virgin Islands, U.S.": "VI",
} 

abbrev_to_us_state = dict(map(reversed, us_state_to_abbrev.items()))
# https://gist.github.com/rogerallen/1583593

In [40]:
data = pd.read_csv('transformed/past_senate_results.csv')
data.head()

,year,state,state_po,special,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct
0,2014,Alabama,AL,False,NaN,795606.0,818090.0,[],Jeff Sessions,[],True,[],381899.0,NaN,NaN,NaN,0.0,381899.00,381899.00,0.000000,100.000000
1,2014,Alaska,AK,False,129431.0,135445.0,282400.0,Mark Begich,Dan Sullivan,True,False,6020171.0,5966351.0,264876.0,48.864752,51.135248,6020171.0,5966351.00,11986522.00,50.224502,49.775498
2,2014,Arkansas,AR,False,334174.0,478819.0,847505.0,Mark L. Pryor,Tom Cotton,True,False,8000604.0,10892504.11,812993.0,41.104167,58.895833,8000604.0,10892504.11,18893108.11,42.346680,57.653320
3,2014,Colorado,CO,False,944203.0,983891.0,2041058.0,Mark Udall,Cory Gardner,True,False,13414189.0,8725449.0,1928094.0,48.970797,51.029203,13414189.0,8725449.00,22139638.00,60.589017,39.410983
4,2014,Delaware,DE,False,130655.0,98823.0,234038.0,Christopher A. Coons,Kevin Wade,True,False,2845059.0,92155.0,229478.0,56.935741,43.064259,2845059.0,92155.00,2937214.00,96.862503,3.137497


In [41]:
demo22_cols = ['state', 'white_pct', 'black_pct', 'hisp_pct', 'asn_pct',
                           'natam_pct', 'pi_pct', 'vap_white_pct', 'vap_black_pct', 'vap_hisp_pct', 'vap_asn_pct',
                           'vap_natam_pct', 'vap_pi_pct', 'cit_white_pct', 'cit_black_pct', 'cit_hisp_pct', 'cit_asn_pct',
                           'cit_natam_pct', 'cit_pi_pct', 'cvap_white_pct', 'cvap_black_pct', 'cvap_hisp_pct', 'cvap_asn_pct',
                           'cvap_natam_pct', 'cvap_pi_pct', 'tot_population', 'white_pop', 'black_pop', 'hisp_pop',
                           'asn_pop', 'natam_pop', 'pi_pop', 'vap_pop', 'white_vap_pop', 'black_vap_pop', 'hisp_vap_pop', 'asn_vap_pop',
                           'natam_vap_pop', 'pi_vap_pop', 'cit_pop', 'white_cit_pop', 'black_cit_pop', 'hisp_cit_pop', 'asn_cit_pop',
                           'natam_cit_pop', 'pi_cit_pop', 'cvap_pop', 'white_cvap_pop', 'black_cvap_pop', 'hisp_cvap_pop', 'asn_cvap_pop',
                           'natam_cvap_pop', 'pi_cvap_pop']

demo_cols = ['state', 'white_pct', 'black_pct', 'hisp_pct', 'aapi_pct',
                           'natam_pct', 'other_pct', 'vap_white_pct', 'vap_black_pct', 'vap_hisp_pct', 'vap_aapi_pct',
                           'vap_natam_pct', 'vap_other_pct', 'cit_white_pct', 'cit_black_pct', 'cit_hisp_pct', 'cit_aapi_pct',
                           'cit_natam_pct', 'cit_other_pct', 'cvap_white_pct', 'cvap_black_pct', 'cvap_hisp_pct', 'cvap_aapi_pct',
                           'cvap_natam_pct', 'cvap_other_pct', 'tot_population', 'white_pop', 'black_pop', 'hisp_pop',
                           'aapi_pop', 'natam_pop', 'other_pop', 'vap_pop', 'white_vap_pop', 'black_vap_pop', 'hisp_vap_pop', 'aapi_vap_pop',
                           'natam_vap_pop', 'other_vap_pop', 'cit_pop', 'white_cit_pop', 'black_cit_pop', 'hisp_cit_pop', 'aapi_cit_pop',
                           'natam_cit_pop', 'other_cit_pop', 'cvap_pop', 'white_cvap_pop', 'black_cvap_pop', 'hisp_cvap_pop', 'aapi_cvap_pop',
                           'natam_cvap_pop', 'other_cvap_pop']

In [60]:
filepath = f'data/demo/2014_114_demo_data_senate/ACSDT5Y2014.B05003D-Data.csv'
demo_b = pd.read_csv(filepath)
demo_b.iloc[0, :].values

array(['Geography', 'Geographic Area Name', 'Estimate!!Total',
       'Margin of Error!!Total', 'Estimate!!Total!!Male',
       'Margin of Error!!Total!!Male',
       'Estimate!!Total!!Male!!Under 18 years',
       'Margin of Error!!Total!!Male!!Under 18 years',
       'Estimate!!Total!!Male!!Under 18 years!!Native',
       'Margin of Error!!Total!!Male!!Under 18 years!!Native',
       'Estimate!!Total!!Male!!Under 18 years!!Foreign born',
       'Margin of Error!!Total!!Male!!Under 18 years!!Foreign born',
       'Estimate!!Total!!Male!!Under 18 years!!Foreign born!!Naturalized U.S. citizen',
       'Margin of Error!!Total!!Male!!Under 18 years!!Foreign born!!Naturalized U.S. citizen',
       'Estimate!!Total!!Male!!Under 18 years!!Foreign born!!Not a U.S. citizen',
       'Margin of Error!!Total!!Male!!Under 18 years!!Foreign born!!Not a U.S. citizen',
       'Estimate!!Total!!Male!!18 years and over',
       'Margin of Error!!Total!!Male!!18 years and over',
       'Estimate!!Total!

In [63]:
demo_b.iloc[0, :].values[18], demo_b.iloc[0, :].values[40] 

('Estimate!!Total!!Male!!18 years and over!!Native',
 'Estimate!!Total!!Female!!18 years and over!!Native')

In [ ]:
demo_key = {
    '': 'tot',
    'B': 'black',
    'C': 'natam',
    'D': 'asn',
    'E': 'pi',
    'F': 'other_only',
    'G': 'multi',
    'H': 'white',
    'I': 'hisp'
}

demo_14 = pd.DataFrame()

for char in demo_key.keys():
    filepath = f'data/demo/2014_114_demo_data_senate/ACSDT5Y2014.B05003{char}-Data.csv'
    demo_b = pd.read_csv(filepath)
    demo_b_ind = demo_b.iloc[1:, 0:2]
    demo_b_native_male = demo_b.iloc[1:, 18]
    demo_b_nonnat_cit_male = demo_b.iloc[1:, 22]
    demo_b_native_female = demo_b.iloc[1:, 40]
    demo_b_nonnat_cit_female = demo_b.iloc[1:, 44]
    if demo_14.empty:
        demo_b = pd.concat([demo_b_ind, demo_b_native_male, demo_b_nonnat_cit_male, demo_b_native_female, demo_b_nonnat_cit_female], axis=1)
        demo_b = demo_b.set_axis(['geoid', 'state', f'cvap_{demo_key[char]}_native_pop_M', f'cvap_{demo_key[char]}_foreign_pop_M',
                                 f'cvap_{demo_key[char]}_native_pop_F', f'cvap_{demo_key[char]}_foreign_pop_F'], axis=1)
        demo_14 = demo_b.copy()
    else:
        demo_b = pd.concat([demo_b_native_male, demo_b_nonnat_cit_male, demo_b_native_female, demo_b_nonnat_cit_female], axis=1)
        demo_b = demo_b.set_axis([f'cvap_{demo_key[char]}_native_pop_M', f'cvap_{demo_key[char]}_foreign_pop_M',
                                 f'cvap_{demo_key[char]}_native_pop_F', f'cvap_{demo_key[char]}_foreign_pop_F'], axis=1)
        demo_14 = pd.concat([demo_14, demo_b], axis=1)

for race in demo_key.values():
    for sex in ['M', 'F']:
        demo_14[f'cvap_{race}_native_pop_{sex}'] = demo_14[f'cvap_{race}_native_pop_{sex}'].astype(int)
        demo_14[f'cvap_{race}_foreign_pop_{sex}'] = demo_14[f'cvap_{race}_foreign_pop_{sex}'].astype(int)
    demo_14[f'cvap_{race}_pop'] = demo_14[f'cvap_{race}_native_pop'] = demo_14[f'cvap_{race}_foreign_pop']
    if race != 'tot':
        demo_14[f'cvap_{race}_pct'] = demo_14[f'cvap_{race}_pop'] / demo_14['cvap_tot_pop'] * 100

demo_14['cvap_aapi_pct'] = demo_14['cvap_asn_pct'] + demo_14['cvap_pi_pct']
demo_14['cvap_other_pct'] = demo_14['cvap_other_only_pct'] + demo_14['cvap_multi_pct']
#demo_14[['seat_number', 'state']] = demo_14['district_name'].str.extract(r"Congressional District (\d+|\(at Large\)) \(114th Congress\), ([A-za-z\-\s]+)")
#demo_14 = demo_14[~demo_14['seat_number'].isna()]
#demo_14['district'] = demo_14['state'].map(us_state_to_abbrev) + '-' + demo_14['seat_number'].map(lambda x: '00' if (isinstance(x, str) and x == '(at Large)') else
#                                                                                                 (f'0{x}' if int(x) < 10 else str(x)))
demo_14.head()

In [42]:
demo_16 = pd.read_csv('data/demo/2016_115_acs_demo_states.csv')
demo_16 = demo_16.iloc[2:]
demo_16 = demo_16.set_axis(demo_cols, axis=1)
demo_16.head()

,state,white_pct,black_pct,hisp_pct,aapi_pct,natam_pct,other_pct,vap_white_pct,vap_black_pct,vap_hisp_pct,vap_aapi_pct,vap_natam_pct,vap_other_pct,cit_white_pct,cit_black_pct,cit_hisp_pct,cit_aapi_pct,cit_natam_pct,cit_other_pct,cvap_white_pct,cvap_black_pct,cvap_hisp_pct,cvap_aapi_pct,cvap_natam_pct,cvap_other_pct,tot_population,white_pop,black_pop,hisp_pop,aapi_pop,natam_pop,other_pop,vap_pop,white_vap_pop,black_vap_pop,hisp_vap_pop,aapi_vap_pop,natam_vap_pop,other_vap_pop,cit_pop,white_cit_pop,black_cit_pop,hisp_cit_pop,aapi_cit_pop,natam_cit_pop,other_cit_pop,cvap_pop,white_cvap_pop,black_cvap_pop,hisp_cvap_pop,aapi_cvap_pop,natam_cvap_pop,other_cvap_pop
2,Alabama,66.3,26.4,4.0,1.3,0.5,1.6,68.5,25.5,3.2,1.3,0.5,1.1,67.5,26.9,2.8,0.8,0.5,1.6,70.0,26.0,1.7,0.8,0.5,1.1,"4,841,160","3,208,550","1,278,745","193,500","63,040","21,960","75,345","3,735,980","2,558,825","951,850","118,885","48,080","17,545","40,785","4,734,505","3,195,135","1,271,600","131,865","39,820","21,910","74,180","3,639,495","2,546,550","945,510","62,375","27,795","17,500",39735
3,Alaska,62.1,3.2,6.7,7.1,13.7,7.3,66.1,3.2,5.8,7.0,12.5,5.4,63.5,3.1,6.4,5.4,14.1,7.5,67.9,3.1,5.4,5.0,13.0,5.6,"736,855","457,540","23,235","49,030","52,365","100,715","53,970","549,240","363,045","17,565","31,785","38,660","68,595","29,590","712,745","452,800","21,855","45,260","38,585","100,690","53,555","527,810","358,490","16,520","28,280","26,625","68,570",29325
4,Arizona,56.3,4.0,30.5,3.2,4.0,2.0,61.3,3.9,26.4,3.3,3.6,1.4,60.0,4.2,27.0,2.4,4.3,2.1,66.4,4.1,21.5,2.5,4.0,1.5,"6,728,580","3,786,195","271,740","2,054,850","213,570","266,975","135,245","5,108,965","3,131,220","200,910","1,348,580","169,235","185,970","73,060","6,192,380","3,713,075","257,705","1,671,880","150,875","266,305","132,560","4,613,575","3,062,780","190,080","991,500","113,155","185,380",70685
5,Arkansas,73.5,15.5,7.0,1.6,0.5,1.9,76.3,14.6,5.5,1.6,0.6,1.4,75.6,15.9,5.0,1.1,0.6,2.0,78.9,15.1,3.0,0.9,0.6,1.4,"2,968,470","2,180,390","458,875","207,055","48,875","16,325","56,980","2,261,240","1,725,465","330,180","125,495","35,935","12,815","31,365","2,874,015","2,171,525","456,850","142,530","30,670","16,295","56,150","2,175,340","1,717,230","328,360","66,205","20,095","12,790",30665
6,California,38.6,5.6,38.6,14.1,0.4,2.7,42.3,5.8,34.4,15.0,0.4,2.1,43.2,6.3,34.2,12.8,0.4,3.1,48.9,6.7,28.0,13.5,0.4,2.4,"38,654,210","14,906,720","2,179,645","14,903,970","5,467,440","138,565","1,057,885","29,513,915","12,484,440","1,700,180","10,166,495","4,435,330","109,655","617,770","33,355,855","14,398,410","2,117,530","11,422,245","4,256,750","137,295","1,023,655","24,582,595","12,021,740","1,644,655","6,891,460","3,327,435","108,530",588785


In [43]:
demo_18 = pd.read_csv('data/demo/2018_116_acs_demo_states.csv')
demo_18 = demo_18.iloc[2:]
demo_18 = demo_18.set_axis(demo_cols, axis=1)
demo_18.head()

,state,white_pct,black_pct,hisp_pct,aapi_pct,natam_pct,other_pct,vap_white_pct,vap_black_pct,vap_hisp_pct,vap_aapi_pct,vap_natam_pct,vap_other_pct,cit_white_pct,cit_black_pct,cit_hisp_pct,cit_aapi_pct,cit_natam_pct,cit_other_pct,cvap_white_pct,cvap_black_pct,cvap_hisp_pct,cvap_aapi_pct,cvap_natam_pct,cvap_other_pct,tot_population,white_pop,black_pop,hisp_pop,aapi_pop,natam_pop,other_pop,vap_pop,white_vap_pop,black_vap_pop,hisp_vap_pop,aapi_vap_pop,natam_vap_pop,other_vap_pop,cit_pop,white_cit_pop,black_cit_pop,hisp_cit_pop,aapi_cit_pop,natam_cit_pop,other_cit_pop,cvap_pop,white_cvap_pop,black_cvap_pop,hisp_cvap_pop,aapi_cvap_pop,natam_cvap_pop,other_cvap_pop
2,Alabama,65.8,26.5,4.2,1.4,0.5,1.7,68.0,25.7,3.3,1.4,0.5,1.2,67.0,26.9,3.0,0.9,0.5,1.7,69.4,26.2,1.9,0.8,0.5,1.2,"4,864,680","3,201,535","1,289,275","203,150","66,755","23,275","80,670","3,765,885","2,561,630","966,735","123,145","51,340","18,800","44,210","4,759,650","3,187,675","1,281,805","144,760","42,480","23,240","79,695","3,671,105","2,548,845","960,070","70,240","29,865","18,765",43305
3,Alaska,61.2,3.1,6.9,7.4,14.0,7.3,65.1,3.2,6.0,7.4,12.8,5.4,62.7,3.0,6.6,5.7,14.5,7.5,67.0,3.1,5.6,5.4,13.3,5.5,"738,515","452,105","22,995","51,185","54,490","103,610","54,130","552,380","359,875","17,625","33,375","40,935","70,850","29,720","713,900","447,265","21,300","47,320","40,705","103,540","53,770","530,385","355,115","16,430","29,915","28,725","70,780",29425
4,Arizona,55.2,4.2,31.1,3.4,3.9,2.2,60.0,4.1,27.2,3.6,3.6,1.6,58.7,4.3,27.9,2.6,4.2,2.3,64.8,4.2,22.7,2.6,4.0,1.7,"6,946,685","3,834,835","288,555","2,163,305","237,485","272,475","150,020","5,312,900","3,188,280","215,405","1,443,570","190,780","192,165","82,680","6,408,360","3,761,305","273,800","1,790,350","163,700","271,705","147,495","4,812,765","3,119,860","204,230","1,092,105","124,550","191,560",80450
5,Arkansas,72.8,15.4,7.3,1.8,0.6,2.2,75.6,14.6,5.9,1.7,0.6,1.6,74.9,15.8,5.4,1.1,0.6,2.2,78.3,15.1,3.4,1.0,0.6,1.6,"2,990,670","2,176,835","459,940","219,055","52,785","17,365","64,700","2,284,725","1,728,370","333,360","133,830","39,345","13,445","36,385","2,893,855","2,168,105","457,480","155,000","32,160","17,365","63,750","2,195,865","1,720,325","331,070","74,000","21,410","13,445",35625
6,California,37.7,5.6,38.9,14.6,0.4,2.9,41.2,5.7,34.9,15.5,0.4,2.2,42.0,6.3,35.1,13.1,0.4,3.2,47.3,6.6,29.2,13.9,0.4,2.5,"39,148,750","14,768,660","2,187,355","15,221,585","5,712,110","140,785","1,118,215","30,075,110","12,402,810","1,724,500","10,509,560","4,668,255","111,145","658,840","33,964,450","14,256,210","2,124,855","11,911,545","4,452,755","139,380","1,079,705","25,232,630","11,940,360","1,668,855","7,374,130","3,511,600","109,940",627735


In [47]:
demo_20 = pd.read_csv('data/demo/2019_117_acs_demo_states.csv')
demo_20 = demo_20.iloc[2:]
demo_20 = pd.concat([demo_20.iloc[:, 0], demo_20.iloc[:, 19:25]], axis=1)
demo_20 = demo_20.set_axis(['state', 'cvap_white_pct', 'cvap_black_pct', 'cvap_hisp_pct', 'cvap_aapi_pct', 'cvap_natam_pct', 'cvap_other_pct'], axis=1)
demo_20.head()

,state,cvap_white_pct,cvap_black_pct,cvap_hisp_pct,cvap_aapi_pct,cvap_natam_pct,cvap_other_pct
2,Alabama,69.2,26.3,2.0,0.9,0.5,1.2
3,Alaska,66.6,3.1,5.7,5.4,13.7,5.5
4,Arizona,64.2,4.3,23.2,2.6,4.0,1.7
5,Arkansas,78.0,15.1,3.6,1.0,0.6,1.7
6,California,46.6,6.6,29.8,14.1,0.4,2.5


In [29]:
demo_22 = pd.read_csv('data/demo/2022_118_acs_demo_states.csv')
demo_22 = demo_22.iloc[2:]
demo_22 = demo_22.set_axis(demo22_cols, axis=1)
demo_22['cvap_aapi_pct'] = demo_22['cvap_asn_pct'].astype(float) + demo_22['cvap_pi_pct'].astype(float) # Combine Asian and Pacific Islander percentages for consistency
demo_22.head()

,state,white_pct,black_pct,hisp_pct,asn_pct,natam_pct,pi_pct,vap_white_pct,vap_black_pct,vap_hisp_pct,vap_asn_pct,vap_natam_pct,vap_pi_pct,cit_white_pct,cit_black_pct,cit_hisp_pct,cit_asn_pct,cit_natam_pct,cit_pi_pct,cvap_white_pct,cvap_black_pct,cvap_hisp_pct,cvap_asn_pct,cvap_natam_pct,cvap_pi_pct,tot_population,white_pop,black_pop,hisp_pop,asn_pop,natam_pop,pi_pop,vap_pop,white_vap_pop,black_vap_pop,hisp_vap_pop,asn_vap_pop,natam_vap_pop,pi_vap_pop,cit_pop,white_cit_pop,black_cit_pop,hisp_cit_pop,asn_cit_pop,natam_cit_pop,pi_cit_pop,cvap_pop,white_cvap_pop,black_cvap_pop,hisp_cvap_pop,asn_cvap_pop,natam_cvap_pop,pi_cvap_pop,cvap_aapi_pct
2,Alabama,65.1,27.3,4.6,1.7,1.1,0.0,67.3,26.2,3.6,1.7,1.2,0.0,66.1,27.7,3.5,1.3,1.1,0.0,68.6,26.7,2.3,1.2,1.2,0.0,"5,028,090","3,271,225","1,370,580","232,405","87,785","56,240","1,590","3,917,450","2,634,490","1,024,660","141,250","65,380","46,045","1,260","4,924,860","3,257,200","1,363,930","173,605","64,645","56,185","1,475","3,824,040","2,621,730","1,019,225","87,570","44,400","45,995","1,160",1.2
3,Alaska,59.3,4.2,7.5,7.7,18.5,1.5,63.3,3.9,6.5,7.5,16.5,1.3,60.5,4.2,7.2,6.4,19.1,1.4,64.8,3.9,6.2,5.9,17.1,1.1,"734,820","436,030","30,980","54,890","56,615","135,985","10,950","555,485","351,895","21,915","36,265","41,870","91,820","7,020","712,100","430,570","29,850","51,190","45,555","135,855","9,795","534,725","346,615","21,035","33,030","31,740","91,685","6,005",7.0
4,Arizona,53.7,5.3,32.0,4.1,4.3,0.2,58.1,4.9,28.4,4.1,4.0,0.2,56.7,5.5,29.3,3.4,4.6,0.2,62.2,5.1,24.6,3.2,4.4,0.2,"7,172,280","3,850,230","381,250","2,297,515","297,185","307,960","12,925","5,578,820","3,243,070","272,780","1,584,280","228,895","224,660","10,200","6,681,080","3,789,020","366,220","1,955,930","226,140","307,595","11,615","5,118,555","3,185,580","261,015","1,259,165","165,105","224,300","9,090",3.4
5,Arkansas,71.5,16.1,8.1,1.9,1.8,0.4,74.3,15.0,6.6,1.8,1.8,0.3,73.6,16.6,6.1,1.4,1.8,0.2,76.9,15.5,4.3,1.3,1.8,0.1,"3,018,670","2,159,400","487,120","243,320","58,135","53,750","11,090","2,321,400","1,725,495","348,780","153,840","42,315","41,145","7,035","2,920,760","2,150,710","484,320","179,480","42,220","53,745","4,755","2,233,470","1,717,850","346,310","94,935","28,600","41,140","2,120",1.4
6,California,35.8,6.2,39.7,16.5,0.8,0.4,39.0,6.2,36.2,16.8,0.9,0.4,39.4,6.9,36.6,15.1,1.0,0.4,44.0,7.1,31.7,15.3,1.0,0.4,"39,356,105","14,099,515","2,450,320","15,617,930","6,489,585","332,895","141,855","30,581,535","11,928,785","1,904,520","11,084,740","5,151,220","265,030","113,270","34,550,270","13,609,700","2,379,290","12,662,405","5,227,415","330,580","123,850","26,078,140","11,485,705","1,841,855","8,266,065","3,997,305","262,900","96,705",15.7


In [34]:
demo_24 = pd.read_csv('data/demo/2024_119_acs_demo_census.csv') # This is CVAP only
demo_24 = pd.pivot_table(demo_24, columns=['lntitle'], index=['geoname', 'geoid'], values='cvap_est', aggfunc='first')
demo_24 = demo_24.reset_index()
extr_tokens = demo_24['geoname'].str.extract(r"Congressional District (\(at Large\)|\d+) \(119th Congress\), ([A-za-z\s]+)")
demo_24 = pd.concat([demo_24, extr_tokens], axis=1)
demo_24 = demo_24.rename({0: 'seat_number', 1: 'state'}, axis=1)
demo_24 = demo_24[~demo_24['state'].isna()]
demo_24['district'] = demo_24['state'].astype(str).map(us_state_to_abbrev) + '-' + demo_24['seat_number'].map(lambda x: '00' if x == '(at Large)' else 
                                                                                                              (f'0{x}' if int(x) < 10 else f'{x}'))
demo_24 = demo_24.set_axis(['geoname', 'geoid', 'natam', 'natamXblack', 'natamXwhite', 'asn', 'asnXwhite', 'black', 'blackXwhite',
                           'hisp', 'pi', 'nothisp', 'multi', 'tot', 'white', 'seat_number', 'state', 'district'], axis=1)
demo_24['aapi'] = demo_24['asn'].astype(int) + demo_24['pi'].astype(int)
demo_24 = demo_24[['state', 'natam', 'aapi', 'black', 'hisp', 'white', 'pi', 'tot']].groupby(['state']).sum()
for race in ['natam', 'aapi', 'black', 'hisp', 'white', 'pi']:
    demo_24[f'cvap_{race}_pct'] = demo_24[race] / demo_24['tot'] * 100
demo_24 = demo_24.rename({'tot': 'cvap_pop'}, axis=1)
demo_24.head()

,natam,aapi,black,hisp,white,pi,cvap_pop,cvap_natam_pct,cvap_aapi_pct,cvap_black_pct,cvap_hisp_pct,cvap_white_pct,cvap_pi_pct
state,,,,,,,,,,,,,
Alabama,11333,39431,989960,106822,2614645,1377,3845692,0.294693,1.025329,25.742051,2.777706,67.988934,0.035806
Alaska,68109,34923,16582,32606,347344,6266,538819,12.640423,6.481397,3.077471,6.051383,64.463948,1.162914
Arizona,177222,155799,236999,1296211,3296328,9045,5324108,3.328670,2.926293,4.451431,24.346069,61.913244,0.169888
Arkansas,8361,26690,330230,110187,1709182,2421,2245935,0.372273,1.188369,14.703453,4.906064,76.101134,0.107795
California,83078,3960026,1640805,8458349,11147879,96710,26132787,0.317907,15.153478,6.278722,32.366808,42.658592,0.370072
